# Seed demo data — `wearables_zerobus` (bronze)

Inserts **Apple HealthKit–shaped** synthetic rows into Unity Catalog bronze:
`users.ankur_nayyar.wearables_zerobus` by default; override with widgets).

**Typical flow:** run this notebook → run your **DLT / Lakeflow** pipeline on silver+gold → open **Lakeview** dashboard (see `dashboards/wearables_apple_health/` and `DASHBOARD_GUIDE.md`).

Rows use `headers.x-device-id = demo-notebook-seed` so you can **replace** only demo rows without touching production ingests.

In [ ]:
dbutils.widgets.text("catalog", "users", "Catalog")
dbutils.widgets.text("schema", "ankur_nayyar", "Schema")
dbutils.widgets.dropdown("payload_size", "demo", ["standard", "demo"], "Payload size")
dbutils.widgets.dropdown(
    "seed_mode",
    "append",
    ["append", "replace_demo_rows"],
    "append = new rows only; replace_demo_rows = delete prior demo-notebook-seed rows for this user then insert",
)
dbutils.widgets.text("user_id", "", "user_id (empty = current_user())")

CATALOG = dbutils.widgets.get("catalog").strip()
SCHEMA = dbutils.widgets.get("schema").strip()
BRONZE = f"{CATALOG}.{SCHEMA}.wearables_zerobus"

uid = dbutils.widgets.get("user_id").strip()
USER_ID = uid if uid else spark.sql("SELECT current_user()").collect()[0][0]

print("Bronze table:", BRONZE)
print("user_id:", USER_ID)
print("payload_size:", dbutils.widgets.get("payload_size"))
print("seed_mode:", dbutils.widgets.get("seed_mode"))

In [ ]:
mode = dbutils.widgets.get("seed_mode")
if mode == "replace_demo_rows":
    spark.sql(f"USE `{CATALOG}`.`{SCHEMA}`")
    n = spark.sql(
        f"""
        SELECT COUNT(*) AS c FROM {BRONZE}
        WHERE user_id = '{USER_ID.replace("'", "''")}'
          AND get_json_object(to_json(headers), '$.x-device-id') = 'demo-notebook-seed'
        """
    ).collect()[0]["c"]
    print(f"Deleting {n} prior demo rows for this user …")
    spark.sql(
        f"""
        DELETE FROM {BRONZE}
        WHERE user_id = '{USER_ID.replace("'", "''")}'
          AND get_json_object(to_json(headers), '$.x-device-id') = 'demo-notebook-seed'
        """
    )
    print("Done.")
else:
    print("seed_mode=append — skipping DELETE.")

In [ ]:
import sys
from pathlib import Path

# Repo / bundle layout: this notebook lives under .../src/notebooks/
nb_path = Path(
    dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get().lstrip("/")
)
src_dir = nb_path.parent.parent  # .../src
ep_dir = src_dir / "endpoint-validation"
sys.path.insert(0, str(ep_dir))
sys.path.insert(0, str(src_dir / "notebooks"))

from bronze_seed_helpers import build_seed_rows

rows = build_seed_rows(USER_ID, dbutils.widgets.get("payload_size"))
print(f"Prepared {len(rows)} bronze rows.")

df = spark.createDataFrame(rows)
df.createOrReplaceTempView("_wearables_bronze_seed_stage")

spark.sql(f"USE `{CATALOG}`.`{SCHEMA}`")
spark.sql(
    f"""
    INSERT INTO {BRONZE} (
      record_id,
      ingested_at,
      body,
      headers,
      record_type,
      source_platform,
      user_id
    )
    SELECT
      record_id,
      ingested_at,
      parse_json(body_json) AS body,
      parse_json(headers_json) AS headers,
      record_type,
      source_platform,
      user_id
    FROM _wearables_bronze_seed_stage
    """
)

inserted = spark.sql(f"SELECT COUNT(*) AS c FROM {BRONZE} WHERE record_id IN (SELECT record_id FROM _wearables_bronze_seed_stage)").collect()[0]["c"]
print(f"Verified {inserted} rows visible in bronze (match stage record_ids).")

In [ ]:
display(
    spark.sql(
        f"""
        SELECT record_type, COUNT(*) AS rows
        FROM {BRONZE}
        WHERE user_id = '{USER_ID.replace("'", "''")}'
          AND get_json_object(to_json(headers), '$.x-device-id') = 'demo-notebook-seed'
        GROUP BY record_type
        ORDER BY record_type
        """
    )
)

## Next steps (Databricks value path)

1. **Run your DLT / Lakeflow pipeline** that reads `wearables_bronze_table` so silver + gold refresh.
2. **Lakeview dashboard:** SQL Editor → open queries under `dashboards/wearables_apple_health/*.sql` → *Save* → *Create dashboard* → add **counter**, **trend**, **bar**, **table** visualizations.
3. **Operational polish:** enable **query tags** in jobs, use **Unity Catalog** lineage on pipeline tables, and (if allowed in your workspace) **Predictive Optimization** on the bronze schema — see `DASHBOARD_GUIDE.md` for a concise checklist.